# Packet Spraying Optimisation: Explore the Results

MEng thesis, Dublin City University · Rachana Santhebachahalli Umesha

This notebook loads the raw ns-3 simulation output in the CSV files in [this repository](https://github.com/RachanaSU/EEN1095_Rachana-S-U) and recreates the
thesis results. It needs no installation: in Google Colab choose **Runtime → Run all**, and every table and chart
appears in about 30 seconds.

Four load-balancing schemes send the same flow over four parallel paths with different delays
(4, 52, 102 and 152 µs):

| Scheme | How it picks a path |
| --- | --- |
| Packet Spraying | Round-robin, every packet on the next path (baseline) |
| PT-ECMP | Sends a *train* of packets on one path, then a drain gap before switching |
| Flowlet Switching | Switches path only when there is an idle gap between bursts |
| LetFlow | Flowlet switching that picks the least-loaded path (advanced simulation only) |

In [ ]:
# Load the data: works locally (after cloning) and in Google Colab
import os, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if os.path.exists("PT-ECMP-results.csv"):
    RES = "."
elif os.path.exists("repo/PT-ECMP-results.csv"):
    RES = "repo"
else:  # running in Colab: fetch the repository
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/RachanaSU/EEN1095_Rachana-S-U", "repo"], check=True)
    RES = "repo"

ALGOS = ["Packet-Spraying", "PT-ECMP", "Flowlet-Switching"]
COLOURS = {"Packet-Spraying": "#E24B4A", "PT-ECMP": "#1D9E75",
           "Flowlet-Switching": "#534AB7", "LetFlow": "#E8871E"}
NAME = {"Packet-Spraying": "Packet Spraying", "PT-ECMP": "PT-ECMP",
        "Flowlet-Switching": "Flowlet Switching", "LetFlow": "LetFlow"}
PATH_LABELS = ["Path 0\n4 µs", "Path 1\n52 µs", "Path 2\n102 µs", "Path 3\n152 µs"]
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def summary(path):
    s = pd.read_csv(path).set_index("metric")["value"]
    return s


def from_trace(path):
    """Recompute the thesis metrics from the per-packet trace (arrival order)."""
    d = pd.read_csv(path)
    seq, delay = d["seq"].to_numpy(), d["delay_ms"].to_numpy()
    seen_max = np.maximum.accumulate(np.r_[-1, seq[:-1]])
    reorder = int((seq < seen_max).sum())          # packet arrives after a later one
    return {"reorder_events": reorder,
            "reorder_%": 100 * reorder / len(seq),
            "avg_delay_ms": delay.mean(),
            "jitter_ms": np.abs(np.diff(delay)).mean()}


print("Loaded", len([f for f in os.listdir(RES) if f.endswith(".csv")]), "result files")

## 1. Spine-leaf and fat-tree results (CBR traffic, 400 packets)

Every metric below is recomputed from the per-packet delay traces, so you can check the thesis numbers yourself.

In [ ]:
rows = []
for topo, prefix in [("Spine-leaf", ""), ("Fat-tree (k=4)", "fattree-")]:
    for a in ALGOS:
        m = from_trace(f"{RES}/{prefix}{a}-delay-trace.csv")
        rows.append({"Topology": topo, "Scheme": a, **m})
table = pd.DataFrame(rows).round({"reorder_%": 2, "avg_delay_ms": 4, "jitter_ms": 4})
table

**Reading it:** packet spraying spreads load perfectly but reorders about 74% of packets, because packets on the
4 µs path overtake those on the 152 µs path. PT-ECMP and flowlet switching both remove reordering completely and cut
jitter by more than 20×. The fat-tree adds a few microseconds of delay (more hops) but shows the same pattern.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 8))
sl = table[table.Topology == "Spine-leaf"].set_index("Scheme")
names = [NAME[a] for a in ALGOS]
cols = [COLOURS[a] for a in ALGOS]

for axis, col, title, unit in [(ax[0, 0], "reorder_%", "(a) Reorder ratio", "%"),
                               (ax[0, 1], "jitter_ms", "(b) Jitter", "ms"),
                               (ax[1, 0], "avg_delay_ms", "(c) Average end-to-end delay", "ms")]:
    vals = sl.loc[ALGOS, col]
    bars = axis.bar(names, vals, color=cols, width=0.55)
    axis.bar_label(bars, labels=[f"{v:.1f}%" if unit == "%" else f"{v:.4f}" for v in vals], fontsize=9)
    axis.set_title(title, fontweight="bold"); axis.set_ylabel(unit)
    axis.set_ylim(0, vals.max() * 1.2 if vals.max() > 0 else 1)

w = 0.26
for i, a in enumerate(ALGOS):
    s = summary(f"{RES}/{a}-results.csv")
    pkts = [int(s[f"path{p}_pkts"]) for p in range(4)]
    ax[1, 1].bar(np.arange(4) + (i - 1) * w, pkts, w, color=COLOURS[a], label=names[i])
ax[1, 1].axhline(100, ls="--", c="grey", lw=1, label="Ideal (100)")
ax[1, 1].set_xticks(range(4), PATH_LABELS, fontsize=8)
ax[1, 1].set_ylim(0, 175)
ax[1, 1].set_title("(d) Packets per path", fontweight="bold"); ax[1, 1].legend(fontsize=8, ncol=2, loc="upper right")
fig.suptitle("Spine-leaf topology, CBR traffic", fontweight="bold")
fig.tight_layout(); plt.show()

## 2. Seeing the reordering

Each dot is one packet: sequence number against the order it arrived in. A perfect diagonal means packets arrived in
order. With packet spraying the dots scatter as faster paths overtake slower ones; with PT-ECMP they sit on the line.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
for axis, a in zip(ax, ALGOS):
    d = pd.read_csv(f"{RES}/{a}-delay-trace.csv").head(80)
    axis.scatter(d["arrival_order"], d["seq"], s=10, color=COLOURS[a])
    axis.plot([0, 80], [0, 80], c="grey", lw=0.8, ls="--")
    axis.set_title(NAME[a], fontweight="bold"); axis.set_xlabel("Arrival order")
ax[0].set_ylabel("Sequence number")
fig.suptitle("First 80 packets: in order on the dashed line", fontweight="bold")
fig.tight_layout(); plt.show()

## 3. Advanced simulation: 4 senders, bursty Pareto traffic, 4 schemes

A harder scenario: four senders and four receivers with bursty Pareto on-off traffic (3,200 packets), adding
LetFlow as a fourth scheme. These figures come from the summary files the simulation writes.

In [ ]:
ADV = ALGOS + ["LetFlow"]
adv = pd.DataFrame({a: summary(f"{RES}/advanced-{a}-results.csv") for a in ADV}).T
adv_table = adv[["total_sent", "total_received", "reorder_events", "reorder_pct", "avg_delay_ms", "avg_jitter_ms"]]
adv_table

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
bars = ax[0].bar([NAME[a] for a in ADV], adv["reorder_pct"].astype(float),
                 color=[COLOURS[a] for a in ADV], width=0.55)
ax[0].bar_label(bars, fmt="%.1f%%", fontsize=9)
ax[0].set_title("Reorder ratio (%)", fontweight="bold"); ax[0].tick_params(axis="x", labelsize=8)
w = 0.2
for i, a in enumerate(ADV):
    pk = [int(adv.loc[a, f"path{p}_pkts"]) for p in range(4)]
    ax[1].bar(np.arange(4) + (i - 1.5) * w, pk, w, color=COLOURS[a], label=NAME[a])
ax[1].axhline(800, ls="--", c="grey", lw=1)
ax[1].set_xticks(range(4), PATH_LABELS, fontsize=8)
ax[1].set_ylim(0, 1200)
ax[1].set_title("Packets per path (ideal 800)", fontweight="bold"); ax[1].legend(fontsize=8, ncol=2, loc="upper right")
fig.tight_layout(); plt.show()

## Conclusion

Across both topologies and both traffic models, packet spraying reorders 64–74% of packets. PT-ECMP, flowlet
switching and LetFlow all deliver every packet in order with near-zero jitter, while still using all four paths.
Flowlet-based schemes keep load closer to even without PT-ECMP's drain-gap cost.

To regenerate the raw data yourself with ns-3, see the [README](https://github.com/RachanaSU/EEN1095_Rachana-S-U#run-the-simulations-ns-3).